In [27]:
import pandas as pd
import numpy as np

customer_info = pd.read_csv('../df_cleaned/customer_info_cln.csv')
item_orders = pd.read_csv('../df_cleaned/item_orders_cln.csv')
order_reviews = pd.read_csv('../df_cleaned/order_reviews_cln.csv')
order_info = pd.read_csv('../df_cleaned/orders_info_cln.csv')
payment = pd.read_csv('../df_cleaned/payment_cln.csv')
product_list = pd.read_csv('../df_cleaned/products_cln.csv')
seller_list = pd.read_csv('../df_cleaned/sellers_cln.csv')

In [28]:
olist_fact_table = item_orders.copy()
olist_fact_table.shape

(112650, 8)

In [29]:
olist_fact_table = olist_fact_table.merge(
    order_info[[
        'order_id', 'customer_id', 'order_status',
        'purchase_ts', 'approved_ts',
        'carrier_delivered_dt', 'customer_delivered_dt',
        'est_delivery_dt', 'status_check', 'sequence_flag',
        'delivery_delay_days'
    ]],
    on='order_id',
    how='left'
)
print("table shape:", olist_fact_table.shape)
olist_fact_table.head()

table shape: (112650, 18)


,Unnamed: 0,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,purchase_ts,approved_ts,carrier_delivered_dt,customer_delivered_dt,est_delivery_dt,status_check,sequence_flag,delivery_delay_days
0,0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,good,ok,-8.0
1,1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,good,ok,-2.0
2,2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,good,ok,-13.0
3,3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20,good,ok,-5.0
4,4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17,good,ok,-15.0


In [30]:
olist_fact_table = olist_fact_table.merge(
    customer_info[[
        'cust_id', 'cust_city', 'cust_state', 'cust_zip_code_prefix'
    ]],
    left_on='customer_id',
    right_on='cust_id'
).drop(columns=['cust_id'])
print("table shape:", olist_fact_table.shape)
olist_fact_table.head()

table shape: (112650, 21)


,Unnamed: 0,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,...,approved_ts,carrier_delivered_dt,customer_delivered_dt,est_delivery_dt,status_check,sequence_flag,delivery_delay_days,cust_city,cust_state,cust_zip_code_prefix
0,0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,3ce436f183e68e07877b285a838db11a,delivered,...,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,good,ok,-8.0,campos dos goytacazes,rj,28013
1,1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,f6dd3ec061db4e3987629fe6b26e5cce,delivered,...,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,good,ok,-2.0,santa fe do sul,sp,15775
2,2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,...,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,good,ok,-13.0,para de minas,mg,35661
3,3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,d4eb9395c8c0431ee92fce09860c5a06,delivered,...,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20,good,ok,-5.0,atibaia,sp,12952
4,4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,58dbd0b2d70206bf40e62cd34e84d795,delivered,...,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17,good,ok,-15.0,varzea paulista,sp,13226


In [32]:
olist_fact_table = olist_fact_table.merge(
    seller_list[[
        'seller_id', 'seller_city', 'seller_state', 'seller_zip_code_prefix'
    ]].rename(columns={
        'seller_zip_code_prefix': 'seller_zip'
    }),
    on='seller_id',
    how='left'
)
print("After sellers join:", olist_fact_table.shape)
olist_fact_table.head()

After sellers join: (112650, 24)


,Unnamed: 0,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,...,est_delivery_dt,status_check,sequence_flag,delivery_delay_days,cust_city,cust_state,cust_zip_code_prefix,seller_city,seller_state,seller_zip
0,0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,3ce436f183e68e07877b285a838db11a,delivered,...,2017-09-29,good,ok,-8.0,campos dos goytacazes,rj,28013,volta redonda,rj,27277
1,1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,f6dd3ec061db4e3987629fe6b26e5cce,delivered,...,2017-05-15,good,ok,-2.0,santa fe do sul,sp,15775,sao paulo,sp,3471
2,2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,...,2018-02-05,good,ok,-13.0,para de minas,mg,35661,borda da mata,mg,37564
3,3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,d4eb9395c8c0431ee92fce09860c5a06,delivered,...,2018-08-20,good,ok,-5.0,atibaia,sp,12952,franca,sp,14403
4,4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,58dbd0b2d70206bf40e62cd34e84d795,delivered,...,2017-03-17,good,ok,-15.0,varzea paulista,sp,13226,loanda,pr,87900


In [34]:
olist_fact_table = olist_fact_table.merge(
    product_list[[
        'product_id', 'product_category_group', 'product_category_name',
        'product_weight_g', 'product_length_cm',
        'product_height_cm', 'product_width_cm'
    ]],
    on='product_id',
    how='left'
)
print("After products join:", olist_fact_table.shape)
olist_fact_table.head()

After products join: (112650, 30)


,Unnamed: 0,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,...,cust_zip_code_prefix,seller_city,seller_state,seller_zip,product_category_group,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,3ce436f183e68e07877b285a838db11a,delivered,...,28013,volta redonda,rj,27277,toys_games_n_hobbies,cool_stuff,650.0,28.0,9.0,14.0
1,1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,f6dd3ec061db4e3987629fe6b26e5cce,delivered,...,15775,sao paulo,sp,3471,home_n_living,pet_shop,30000.0,50.0,30.0,40.0
2,2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,...,35661,borda da mata,mg,37564,home_n_living,furniture_decor,3050.0,33.0,13.0,33.0
3,3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,d4eb9395c8c0431ee92fce09860c5a06,delivered,...,12952,franca,sp,14403,health_n_beauty,perfumery,200.0,16.0,10.0,15.0
4,4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,58dbd0b2d70206bf40e62cd34e84d795,delivered,...,13226,loanda,pr,87900,home_n_living,garden_tools,3750.0,35.0,40.0,30.0


In [40]:
# ── PRIMARY PAYMENT METHOD (for fact_orders_flat) ──────────
primary_payment = payment[
    payment['payment_sequential'] == 1
][['order_id', 'payment_type']].rename(columns={
    'payment_type': 'primary_payment_method'
})

# ── PIVOT UP TO 3 PAYMENT METHODS (for fact_orders_flags) ──
payments_pivot = payment[
    payment['payment_sequential'] <= 3
].pivot_table(
    index='order_id',
    columns='payment_sequential',
    values='payment_type',
    aggfunc='first'
).reset_index()

payments_pivot.columns.name = None
payments_pivot = payments_pivot.rename(columns={
    1: 'payment_method_1',
    2: 'payment_method_2',
    3: 'payment_method_3'
})

# ensure all 3 columns exist
for col in ['payment_method_1', 'payment_method_2', 'payment_method_3']:
    if col not in payments_pivot.columns:
        payments_pivot[col] = None

# ── AGGREGATE NUMERIC COLUMNS ───────────────────────────────
payments_agg = payment.groupby('order_id').agg(
    total_payment_value=('payment_value', 'sum'),
    payment_installments=('payment_installments', 'max'),
    payment_methods_count=('payment_sequential', 'max')
).reset_index()

# ── FLAGS (for fact_orders_flags) ──────────────────────────
payments_agg['is_mixed_payment'] = payments_agg['payment_methods_count'] > 1

payments_agg['is_multi_voucher'] = (
    payment[payment['payment_type'] == 'voucher']
    .groupby('order_id')['payment_sequential']
    .count()
    .reset_index()
    .rename(columns={'payment_sequential': 'voucher_count'})
    .set_index('order_id')['voucher_count'] > 1
).reindex(payments_agg['order_id']).fillna(False).values

# ── COMBINE EVERYTHING ──────────────────────────────────────
payments_final = payments_agg.merge(
    primary_payment, on='order_id', how='left'
).merge(
    payments_pivot, on='order_id', how='left'
)

print(payments_final.shape)
print(payments_final.columns.tolist())
print(payments_final.head())

(99440, 10)
['order_id', 'total_payment_value', 'payment_installments', 'payment_methods_count', 'is_mixed_payment', 'is_multi_voucher', 'primary_payment_method', 'payment_method_1', 'payment_method_2', 'payment_method_3']
                           order_id  total_payment_value  \
0  00010242fe8c5a6d1ba2dd792cb16214                72.19   
1  00018f77f2f0320c557190d7a144bdd3               259.83   
2  000229ec398224ef6ca0657da4fc703e               216.87   
3  00024acbcdf0a6daa1e931b038114c75                25.78   
4  00042b26cf59d7ce69dfabb4e55b4fd9               218.04   

   payment_installments  payment_methods_count  is_mixed_payment  \
0                     2                      1             False   
1                     3                      1             False   
2                     5                      1             False   
3                     2                      1             False   
4                     3                      1             False   

  is_mu

In [43]:
olist_fact_table = olist_fact_table.merge(
    payments_final[[
        'order_id', 'total_payment_value', 'payment_installments',
        'payment_methods_count', 'is_mixed_payment', 'is_multi_voucher',
        'primary_payment_method', 'payment_method_1', 'payment_method_2',
        'payment_method_3' 
    ]],
    on='order_id',
    how='left'
)
print("After payments join:", olist_fact_table.shape)
olist_fact_table.head()

After payments join: (112650, 39)


,Unnamed: 0,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,...,product_width_cm,total_payment_value,payment_installments,payment_methods_count,is_mixed_payment,is_multi_voucher,primary_payment_method,payment_method_1,payment_method_2,payment_method_3
0,0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,3ce436f183e68e07877b285a838db11a,delivered,...,14.0,72.19,2.0,1.0,False,False,credit card,credit card,NaN,NaN
1,1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,f6dd3ec061db4e3987629fe6b26e5cce,delivered,...,40.0,259.83,3.0,1.0,False,False,credit card,credit card,NaN,NaN
2,2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,...,33.0,216.87,5.0,1.0,False,False,credit card,credit card,NaN,NaN
3,3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,d4eb9395c8c0431ee92fce09860c5a06,delivered,...,15.0,25.78,2.0,1.0,False,False,credit card,credit card,NaN,NaN
4,4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,58dbd0b2d70206bf40e62cd34e84d795,delivered,...,30.0,218.04,3.0,1.0,False,False,credit card,credit card,NaN,NaN


In [44]:
olist_fact_table = olist_fact_table.merge(
    order_reviews[[
        'order_id', 'review_score', 'review_creation_date'
    ]].drop_duplicates(subset='order_id'),  # keep one review per order
    on='order_id',
    how='left'
)
print("After reviews join:", olist_fact_table.shape)
olist_fact_table.head()

After reviews join: (112650, 41)


,Unnamed: 0,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,...,payment_installments,payment_methods_count,is_mixed_payment,is_multi_voucher,primary_payment_method,payment_method_1,payment_method_2,payment_method_3,review_score,review_creation_date
0,0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,3ce436f183e68e07877b285a838db11a,delivered,...,2.0,1.0,False,False,credit card,credit card,NaN,NaN,5.0,2017-09-21 00:00:00
1,1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,f6dd3ec061db4e3987629fe6b26e5cce,delivered,...,3.0,1.0,False,False,credit card,credit card,NaN,NaN,4.0,2017-05-13 00:00:00
2,2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,...,5.0,1.0,False,False,credit card,credit card,NaN,NaN,5.0,2018-01-23 00:00:00
3,3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,d4eb9395c8c0431ee92fce09860c5a06,delivered,...,2.0,1.0,False,False,credit card,credit card,NaN,NaN,4.0,2018-08-15 00:00:00
4,4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,58dbd0b2d70206bf40e62cd34e84d795,delivered,...,3.0,1.0,False,False,credit card,credit card,NaN,NaN,5.0,2017-03-02 00:00:00


In [45]:
olist_fact_table.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 41 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   Unnamed: 0              112650 non-null  int64  
 1   order_id                112650 non-null  str    
 2   order_item_id           112650 non-null  int64  
 3   product_id              112650 non-null  str    
 4   seller_id               112650 non-null  str    
 5   shipping_limit_date     112650 non-null  str    
 6   price                   112650 non-null  float64
 7   freight_value           112650 non-null  float64
 8   customer_id             112650 non-null  str    
 9   order_status            112650 non-null  str    
 10  purchase_ts             112650 non-null  str    
 11  approved_ts             112650 non-null  str    
 12  carrier_delivered_dt    111456 non-null  str    
 13  customer_delivered_dt   110196 non-null  str    
 14  est_delivery_dt         112650 

In [47]:
# ── FACT_ORDERS_FLAT (core analysis table for Power BI) ────
olist_orders_fact= olist_fact_table[[
    'order_id',
    'order_item_id',
    'product_id',
    'seller_id',
    'customer_id',
    'order_status',
    'product_category_group',
    'product_category_name',
    'price',
    'freight_value',
    'total_payment_value',
    'payment_installments',
    'primary_payment_method',
    'review_score',
    'purchase_ts',
    'approved_ts',
    'carrier_delivered_dt',
    'customer_delivered_dt',
    'est_delivery_dt',
    'shipping_limit_date',
    'cust_city',
    'cust_state',
    'seller_city',
    'seller_state',
    'delivery_delay_days'
]]

# ── FACT_ORDERS_FLAGS (audit/data quality table) ───────────
olist_orders_flags = olist_fact_table[[
    'order_id',
    'order_item_id',
    'status_check',
    'sequence_flag',
    'is_mixed_payment',
    'is_multi_voucher',
    'payment_methods_count',
    'payment_method_1',
    'payment_method_2',
    'payment_method_3',
    'review_creation_date'
]]

print("fact_orders_flat shape:", olist_orders_fact.shape)
print("fact_orders_flags shape:", olist_orders_flags.shape)
print("\nfact_orders_flat columns:", olist_orders_fact.columns.tolist())
print("\nfact_orders_flags columns:", olist_orders_flags.columns.tolist())

fact_orders_flat shape: (112650, 25)
fact_orders_flags shape: (112650, 11)

fact_orders_flat columns: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'customer_id', 'order_status', 'product_category_group', 'product_category_name', 'price', 'freight_value', 'total_payment_value', 'payment_installments', 'primary_payment_method', 'review_score', 'purchase_ts', 'approved_ts', 'carrier_delivered_dt', 'customer_delivered_dt', 'est_delivery_dt', 'shipping_limit_date', 'cust_city', 'cust_state', 'seller_city', 'seller_state', 'delivery_delay_days']

fact_orders_flags columns: ['order_id', 'order_item_id', 'status_check', 'sequence_flag', 'is_mixed_payment', 'is_multi_voucher', 'payment_methods_count', 'payment_method_1', 'payment_method_2', 'payment_method_3', 'review_creation_date']


In [ ]:
dim_customers = customer_info[[
    'customer_id',
    'customer_unique_id',
    'cust_zip_code_prefix',
    'cust_city',
    'cust_state'
]].drop_duplicates()

print(dim_customers.shape)
dim_customers.head()

In [ ]:
dim_products = products[[
    'product_id',
    'product_category_name',
    'product_name_length',
    'product_description_length',
    'product_photos_qty',
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm'
]].drop_duplicates()

print(dim_products.shape)
dim_products.head()

In [ ]:
dim_sellers = sellers[[
    'seller_id',
    'seller_zip_code_prefix',
    'seller_city',
    'seller_state'
]].drop_duplicates()

# validate
print("dim_sellers shape:", dim_sellers.shape)
print("seller_id unique:", dim_sellers['seller_id'].is_unique)
print("Nulls:\n", dim_sellers.isnull().sum())

In [48]:
customer_info.head()

,Unnamed: 0,cust_id,cust_unique_id,cust_zip_code_prefix,cust_city,cust_state
0,0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,sp
1,1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,sp
2,2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,sp
3,3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,sp
4,4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,sp


RAW dim_date

In [ ]:
# use your fact table's actual date range
# don't hardcode — derive it from the data itself
date_min = fact_orders_flat['purchase_ts'].min()
date_max = fact_orders_flat['purchase_ts'].max()

print("Date range:", date_min, "to", date_max)

In [ ]:
# create one row per calendar date covering the full range
dim_dates = pd.DataFrame({
    'date': pd.date_range(start=date_min.date(), 
                          end=date_max.date(), 
                          freq='D')
})

print("Total days generated:", len(dim_dates))
dim_dates.head()

In [ ]:
dim_dates['year'] = dim_dates['date'].dt.year
dim_dates['month'] = dim_dates['date'].dt.month
dim_dates['month_name'] = dim_dates['date'].dt.strftime('%B')  # January, February...
dim_dates['quarter'] = dim_dates['date'].dt.quarter
dim_dates['quarter_label'] = 'Q' + dim_dates['quarter'].astype(str)  # Q1, Q2, Q3, Q4
dim_dates['week_of_year'] = dim_dates['date'].dt.isocalendar().week.astype(int)
dim_dates['day_of_week'] = dim_dates['date'].dt.dayofweek  # 0=Monday, 6=Sunday
dim_dates['day_name'] = dim_dates['date'].dt.strftime('%A')  # Monday, Tuesday...
dim_dates['is_weekend'] = dim_dates['day_of_week'].isin([5, 6])  # Saturday=5, Sunday=6
dim_dates['year_month'] = dim_dates['date'].dt.strftime('%Y-%m')  # 2018-01 (useful for grouping)
dim_dates['year_quarter'] = dim_dates['date'].dt.year.astype(str) + '-' + dim_dates['quarter_label']  # 2018-Q1

print(dim_dates.shape)
dim_dates.head(10)

In [ ]:
# YYYYMMDD format — e.g., 20180101
# useful as a join key since integers join faster than datetime strings
dim_dates['date_key'] = dim_dates['date'].dt.strftime('%Y%m%d').astype(int)

print(dim_dates.columns.tolist())
dim_dates.head()

In [ ]:
# add date_key to fact table so Power BI can join on it
fact_orders_flat['purchase_date_key'] = (
    fact_orders_flat['purchase_ts']
    .dt.strftime('%Y%m%d')
    .astype(float)  # handles NaT gracefully
    .astype('Int64')  # nullable integer
)

print("date_key added to fact table")
fact_orders_flat[['purchase_ts', 'purchase_date_key']].head()

In [ ]:
# validate
print("dim_dates shape:", dim_dates.shape)
print("Date range:", dim_dates['date'].min(), "to", dim_dates['date'].max())
print("Any nulls:", dim_dates.isnull().sum().sum())
print("date_key unique:", dim_dates['date_key'].is_unique)

# save
dim_dates.to_csv('../csv-files/model/dim_dates.csv', index=False)
print("✅ dim_dates saved")

# also resave fact_orders_flat with the new date_key column
fact_orders_flat.to_csv('../csv-files/model/fact_orders_flat.csv', index=False)
print("✅ fact_orders_flat resaved with purchase_date_key")